# ARC-AGI-2 - First real Kaggle submission (symbolic solver only)

See `docs/decisions/0047-primeira-submissao-real-kaggle.md`.

**Purpose:** validate that the real Kaggle environment works end to end -
notebook execution, input file discovery, `submission.json` format
(ADR 0006), no-internet execution, and total run time - using the
symbolic solver only (geometric transforms + color mapping,
`src/solvers/baseline_solver.py`). No GPU, no neural model, no TTT.

**Not the goal:** accuracy. ADR 0040-0046 already established this
solver's held-out coverage is near zero on two independent 40-task
samples; that is expected and not a bug in this notebook.

**Self-contained by design:** every function below is inlined rather
than imported from this repo, so the notebook has zero dependency on
attaching this project as a Kaggle Dataset and zero pip installs (it
only ever touches Python's standard library). Logic mirrors, and must
be kept in sync with, `src/utils/grid_ops.py`, `src/solvers/color_mapping.py`,
`src/solvers/baseline_solver.py`, `src/evaluation/submission_format.py`
and `src/evaluation/submission_fallback.py`.

**This notebook never calls the Kaggle submission API.** Producing
`submission.json` here is the last automated step; a human reviews it
and clicks Submit in the Kaggle UI.

## 1. Locate the competition's input file

No hardcoded competition slug: walks `/kaggle/input` looking for the
official `*test_challenges.json` file (ADR 0006 names
`arc-agi_test_challenges.json`). Falls back to a local synthetic file
so this same notebook can be smoke-tested outside Kaggle.

In [ ]:
import json
import os
import time
from collections import Counter


def find_challenges_path():
    kaggle_input = "/kaggle/input"
    if os.path.isdir(kaggle_input):
        for root, _, files in os.walk(kaggle_input):
            for fname in files:
                if fname.endswith("test_challenges.json"):
                    return os.path.join(root, fname)
    local_fallback = "outputs/_diagnostics/synthetic_arc-agi_test_challenges.json"
    if os.path.exists(local_fallback):
        print(f"/kaggle/input not found or no match; using local fallback for testing: {local_fallback}")
        return local_fallback
    raise FileNotFoundError(
        "No *test_challenges.json found under /kaggle/input and no local fallback present."
    )


CHALLENGES_PATH = find_challenges_path()
OUTPUT_PATH = "submission.json" if not os.path.isdir("/kaggle/working") else "/kaggle/working/submission.json"
print(f"Challenges file: {CHALLENGES_PATH}")
print(f"Submission will be written to: {OUTPUT_PATH}")

## 2. Geometric transforms

Mirrors `src/utils/grid_ops.py`: the 8 elements of the D4 dihedral group
(identity, 3 rotations, 2 axis flips, 2 diagonal reflections).

In [ ]:
def identity(grid):
    return [row[:] for row in grid]


def rotate90(grid):
    return [list(row) for row in zip(*grid[::-1])]


def rotate180(grid):
    return [row[::-1] for row in grid[::-1]]


def rotate270(grid):
    return [list(row) for row in zip(*grid)][::-1]


def flip_horizontal(grid):
    return [row[::-1] for row in grid]


def flip_vertical(grid):
    return grid[::-1]


def transpose(grid):
    return [list(row) for row in zip(*grid)]


def anti_transpose(grid):
    return rotate180(transpose(grid))


GEOMETRIC_TRANSFORMS = [
    identity,
    rotate90,
    rotate180,
    rotate270,
    flip_horizontal,
    flip_vertical,
    transpose,
    anti_transpose,
]

## 3. Color mapping

Mirrors `src/solvers/color_mapping.py`: a single, exact, position-verified
cell-wise color substitution, only defined when every train pair shares
input/output shape and never conflicts.

In [ ]:
def same_shape(a, b):
    return len(a) == len(b) and all(len(ra) == len(rb) for ra, rb in zip(a, b))


def infer_color_mapping(train_pairs):
    mapping = {}
    for input_grid, output_grid in train_pairs:
        if not same_shape(input_grid, output_grid):
            return None
        for in_row, out_row in zip(input_grid, output_grid):
            for in_cell, out_cell in zip(in_row, out_row):
                if in_cell in mapping and mapping[in_cell] != out_cell:
                    return None
                mapping[in_cell] = out_cell
    return mapping


def apply_color_mapping(grid, mapping):
    return [[mapping.get(cell, cell) for cell in row] for row in grid]

## 4. Symbolic solver

Mirrors `src/solvers/baseline_solver.py`: try every geometric transform
that exactly fits all train pairs, then a global color mapping, capped
at 2 candidate predictions per test pair (ADR 0006's `attempt_1`/`attempt_2`).

In [ ]:
MAX_PREDICTIONS = 2


def _transform_fits_all_train_pairs(transform, train_pairs):
    return all(transform(i) == o for i, o in train_pairs)


def _find_matching_geometric_transforms(train_pairs):
    return [t for t in GEOMETRIC_TRANSFORMS if _transform_fits_all_train_pairs(t, train_pairs)]


def predict_test_input(train_pairs, test_input):
    predictions = []

    for transform in _find_matching_geometric_transforms(train_pairs):
        candidate = transform(test_input)
        if candidate not in predictions:
            predictions.append(candidate)
        if len(predictions) >= MAX_PREDICTIONS:
            return predictions[:MAX_PREDICTIONS]

    mapping = infer_color_mapping(train_pairs)
    if mapping is not None:
        candidate = apply_color_mapping(test_input, mapping)
        if candidate not in predictions:
            predictions.append(candidate)

    return predictions[:MAX_PREDICTIONS]


def solve_task(task):
    train_pairs = [(p["input"], p["output"]) for p in task["train"]]
    return [predict_test_input(train_pairs, p["input"]) for p in task["test"]]

## 5. Submission format + safety-net fallback

Mirrors `src/evaluation/submission_format.py` (ADR 0006) and
`src/evaluation/submission_fallback.py` (ADR 0011): every task_id must
appear, both attempts are always present, and a task with zero
candidate predictions falls back to the test input copy plus the most
common (unambiguous) train output, never an empty/placeholder grid.

In [ ]:
def _grid_key(grid):
    return tuple(tuple(row) for row in grid)


def _most_common_train_output(train_pairs):
    if not train_pairs:
        return None
    outputs = [o for _, o in train_pairs]
    output_by_key = {_grid_key(o): o for o in outputs}
    counts = Counter(_grid_key(o) for o in outputs)
    ranked = counts.most_common()
    top_key, top_count = ranked[0]
    if top_count <= 1:
        return None
    if len(ranked) > 1 and ranked[1][1] == top_count:
        return None
    return output_by_key[top_key]


def _build_attempt_pair(predictions, test_input, train_pairs):
    if not predictions:
        attempt_1 = test_input
        most_common = _most_common_train_output(train_pairs)
        attempt_2 = most_common if most_common is not None else attempt_1
    else:
        attempt_1 = predictions[0]
        attempt_2 = predictions[1] if len(predictions) > 1 else attempt_1
    return {"attempt_1": attempt_1, "attempt_2": attempt_2}


def build_submission(tasks):
    submission = {}
    for task_id, task in tasks.items():
        train_pairs = [(p["input"], p["output"]) for p in task["train"]]
        predictions_per_pair = solve_task(task)
        submission[task_id] = [
            _build_attempt_pair(predictions, test_pair["input"], train_pairs)
            for predictions, test_pair in zip(predictions_per_pair, task["test"])
        ]
    return submission


def _validate_grid(task_id, grid):
    if not isinstance(grid, list) or not grid or not all(isinstance(row, list) for row in grid):
        raise ValueError(f"task {task_id}: grid must be a non-empty list of lists, got {grid!r}")
    row_length = len(grid[0])
    for row in grid:
        if len(row) != row_length:
            raise ValueError(f"task {task_id}: grid rows must all have the same length")
        if not all(isinstance(cell, int) and 0 <= cell <= 9 for cell in row):
            raise ValueError(f"task {task_id}: grid cells must be integers 0-9")


def validate_submission(submission, tasks):
    missing = set(tasks) - set(submission)
    if missing:
        raise ValueError(f"submission missing task_id(s): {sorted(missing)}")
    for task_id, task in tasks.items():
        entries = submission[task_id]
        expected_count = len(task["test"])
        if len(entries) != expected_count:
            raise ValueError(
                f"task {task_id}: expected {expected_count} test entries, got {len(entries)}"
            )
        for entry in entries:
            if set(entry) != {"attempt_1", "attempt_2"}:
                raise ValueError(f"task {task_id}: entry must have exactly attempt_1/attempt_2, got {set(entry)}")
            for grid in entry.values():
                _validate_grid(task_id, grid)

## 6. Run: load, solve, build, validate, write

In [ ]:
with open(CHALLENGES_PATH, "r", encoding="utf-8") as f:
    tasks = json.load(f)
print(f"Loaded {len(tasks)} tasks from {CHALLENGES_PATH}")

start = time.monotonic()
submission = build_submission(tasks)
elapsed = time.monotonic() - start

validate_submission(submission, tasks)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(submission, f)

print(f"OK: {len(tasks)} tasks solved and validated in {elapsed:.3f}s -> {OUTPUT_PATH}")
if len(tasks) > 0:
    print(f"({elapsed / len(tasks) * 1000:.2f} ms/task; symbolic solver only, no GPU used)")

## 7. Sanity summary (informational only)

Non-empty prediction counts, so a reviewer can see how often the
symbolic solver produced a real candidate versus fell back to the
ADR 0011 safety net, before deciding whether to submit.

In [ ]:
real_prediction_pairs = 0
fallback_pairs = 0
total_pairs = 0

for task_id, task in tasks.items():
    train_pairs = [(p["input"], p["output"]) for p in task["train"]]
    for test_pair in task["test"]:
        total_pairs += 1
        predictions = predict_test_input(train_pairs, test_pair["input"])
        if predictions:
            real_prediction_pairs += 1
        else:
            fallback_pairs += 1

print(f"Total test pairs: {total_pairs}")
print(f"  solved by a real symbolic candidate: {real_prediction_pairs}")
print(f"  fell back to ADR 0011 safety net:    {fallback_pairs}")
print()
print("Reminder: this notebook does NOT call the Kaggle submission API.")
print(f"Review {OUTPUT_PATH} and submit manually from the Kaggle UI when ready.")

---

# Consolidation (2026-09-16): hybrid pipeline added to this notebook

Everything above this point (Sections 1-7) is unchanged from the version
already pushed and submitted for real on Kaggle (ref 56256382, ADR 0047).

Parts B-F below add the offline neural-model packaging validated in
`notebooks/offline_model_validation/` (ADR 0048) and the time-budgeted
hybrid symbolic+neural pipeline designed and locally validated in
ADR 0049, consolidated into this single notebook so the whole pipeline
runs on the one Kaggle kernel already confirmed to have `GPU L4 x4`
available and selected, instead of debugging a second, separate notebook
that was not offering that accelerator.

**Plan (see the consolidation prompt, 2026-09-16):**
- Part B: offline packaging (OLMo-2-1124-7B 4-bit + Unsloth/bitsandbytes/etc), confirmed working with no internet, ported from the already-validated diagnostic notebook.
- Part C: the neural solver's actual production path (TTT + self-consistency + generation), mirroring `src/solvers/neural_solver.py` exactly - the diagnostic-only shape/decode mitigations (ADR 0025-0032) are intentionally not ported, since they were never wired into production either.
- Part D: the hybrid time-budget pipeline (ADR 0049).
- Part E: a small (1-2 task) real-GPU timing test on this specific L4x4 hardware, to measure real per-task time before spending the scarce, shared 29h/week L4x4 quota on a full run.
- Part F: the full hybrid pipeline run, gated behind a flag left `False` until Part E's timing is reported and reviewed.

**Standing rule, unchanged:** this notebook never calls the Kaggle
submission API. No real submission happens at any stage without the
user's own separate, explicit approval given in chat.

**Final consolidation update (2026-09-18):** per ADR 0059's cross-task
pretraining pilot (excluded from this pipeline, insufficient evidence)
and the standing decision to ship a final submission, the base model is
`Qwen/Qwen3-4B-Base` (ADR 0051/0055/0056), not OLMo-2-1124-7B as Part B
was originally written against. Part B's dataset source is now
`arc-agi2-qwen3-4b-base-4bit-adr0049` (still resolved dynamically by
keyword, not hardcoded), and Part C's generation cell carries all 4
validated ADR 0056 failure-mode mitigations (hallucinated second
example, degenerate line repetition, topic drift, EOS/stopping) plus
the corrected parser (a completion matching any of those 3 patterns is
never counted as `kept`). This is the final submission run: no
cross-task pretraining warm-start anywhere in this pipeline.

## Part B - Offline neural-model packaging (ADR 0048)

Resolves the two private Kaggle Datasets attached to this kernel
(`arc-agi2-offline-wheelhouse-adr0048`, `arc-agi2-qwen3-4b-base-4bit-adr0049`),
installs the 5 missing packages fully offline, and confirms there is no
network access, exactly as already validated end to end in the separate
`offline_model_validation` notebook. **Lesson learned there (real Kaggle
finding):** private datasets mount at
`/kaggle/input/datasets/<owner-username>/<dataset-slug>/`, not
`/kaggle/input/<slug>/`, so paths are resolved dynamically below, never
hardcoded.

In [ ]:
import subprocess
import socket
import urllib.request
import importlib


def _resolve_dataset_dir(keyword):
    base = "/kaggle/input"
    for root, dirs, _ in os.walk(base):
        for d in dirs:
            if keyword.lower() in d.lower():
                return os.path.join(root, d)
    raise FileNotFoundError(f"No directory under {base} matching keyword '{keyword}'")


WHEELHOUSE_DIR = _resolve_dataset_dir("wheelhouse")
MODEL_DIR = _resolve_dataset_dir("qwen3")
print(f"Wheelhouse resolved to: {WHEELHOUSE_DIR}")
print(f"Model dir resolved to:  {MODEL_DIR}")

_OFFLINE_PACKAGES = ["unsloth", "unsloth_zoo", "bitsandbytes", "trl", "xformers"]

_install_result = subprocess.run(
    ["pip", "install", "--no-index", "--no-deps", "--find-links", WHEELHOUSE_DIR] + _OFFLINE_PACKAGES,
    capture_output=True,
    text=True,
)
print(_install_result.stdout[-2000:])
print(_install_result.stderr[-2000:])
assert _install_result.returncode == 0, "Offline pip install failed, see output above"
print("Offline pip install: OK (returncode 0)")

In [ ]:
socket.setdefaulttimeout(5)
try:
    urllib.request.urlopen("https://huggingface.co")
    raise AssertionError("Internet appears reachable; expected enable_internet=false")
except AssertionError:
    raise
except Exception as e:
    print(f"Internet unreachable as expected: {type(e).__name__}: {e}")

for _pkg in _OFFLINE_PACKAGES:
    _mod = importlib.import_module(_pkg)
    print(f"{_pkg}: {getattr(_mod, '__version__', 'unknown version')}")

## Part C - Neural solver core (production path only, ADR 0003/0014/0033)

Mirrors `src/solvers/neural_solver.py` and `src/solvers/neural/*.py`
exactly, adapted from `Task`/`Pair` NamedTuples to this notebook's plain
dict/tuple task representation. Deliberately **excludes** the
diagnostic-only mitigations from ADR 0025-0031 (shape constraint,
fixed-shape rule): those were confirmed by direct code reading to never
have been wired into the real `neural_solver.solve_task` production
path, so porting them here would add scope this consolidation does not
need.

**2026-09-16 ADR 0032 reactivation:** the per-attempt conditional
`no_repeat_ngram_size` escalation (ADR 0032) is now ported into
`generate_grid_predictions` below, mirroring the same fix applied to
`src/solvers/neural/generation.py`. Root cause of it being inactive: it
was implemented and validated only inside a parallel diagnostic module
(`per_attempt_conditional_generation_diagnostics.py`), never merged into
the production `generate_grid_predictions` function itself, and no
config flag for it existed in `NeuralSolverConfig` (see the new
`enable_conditional_escalation` field below, default `True`). Config
knobs: geometric augmentation on, color augmentation off,
`repetition_penalty=1.0`/`no_repeat_ngram_size=0` (baseline decoding,
escalates to `no_repeat_ngram_size=3` per attempt on detection),
matching the production defaults in `src/solvers/neural/config.py`.

`MODEL_NAME` is overridden to the resolved local `MODEL_DIR` from Part B
(the offline 4-bit checkpoint), since this environment has no internet
access to the Hugging Face Hub.


**2026-09-16 ADR 0049 circuit breaker port:** a per-task neural time
ceiling (`NEURAL_TASK_CEILING_SECONDS = 400.0`) is now ported into this
Part C, mirroring `src/evaluation/task_time_limit.py`,
`src/solvers/neural/deadline_stopping_criteria.py`, and
`src/solvers/neural/deadline_callback.py` exactly (same drift risk the
ADR 0032 reactivation note above already documented for this notebook).
If a single task's TTT+generation runs past the ceiling,
`TaskTimeExceeded` aborts that task's neural processing and
`neural_solve_task` falls back to the symbolic solver, exactly like any
other neural failure; `TIME_LIMIT_ABORTS` counts how often this happens
across a run. This does not diagnose why any given task ran long (see
`264363fd`'s still-undiagnosed outlier, round 3 above); it only bounds
the damage a single anomalous task can do to the shared neural-pass
budget.

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional, Tuple


@dataclass
class NeuralSolverConfig:
    model_name: str = MODEL_DIR
    load_in_4bit: bool = True
    max_seq_length: int = 2048

    lora_rank: int = 16
    lora_alpha: int = 16
    lora_dropout: float = 0.0
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ])

    ttt_learning_rate: float = 1e-4
    ttt_num_epochs: int = 3

    use_geometric_augmentation: bool = True
    use_color_augmentation: bool = False
    num_color_augmentations_per_pair: int = 2
    color_augmentation_seed: int = 42

    max_new_tokens: int = 1024
    num_predictions: int = 2
    generation_temperature: float = 0.7
    repetition_penalty: float = 1.0
    no_repeat_ngram_size: int = 0
    stop_on_second_input: bool = False

    # ADR 0032: every generation attempt is checked for the degenerate
    # pattern (see _shows_degenerate_pattern below); on detection, that
    # call's remaining attempts escalate one-way to no_repeat_ngram_size=3.
    enable_conditional_escalation: bool = True

    checkpoint_output_dir: str = "/kaggle/working/ttt_tmp"
    checkpoint_save_strategy: str = "no"
    checkpoint_save_steps: int = 50
    checkpoint_save_total_limit: int = 2


NEURAL_CONFIG = NeuralSolverConfig()
print(f"Neural solver config ready, model_name={NEURAL_CONFIG.model_name}, enable_conditional_escalation={NEURAL_CONFIG.enable_conditional_escalation}")


In [ ]:
PROMPT_PREFIX = "Input:\n"
COMPLETION_PREFIX = "\nOutput:\n"


def grid_to_text(grid):
    return "\n".join("".join(str(cell) for cell in row) for row in grid)


def _parse_row(line):
    if not line or not all(ch.isdigit() for ch in line):
        return None
    return [int(ch) for ch in line]


def _is_rectangular(rows):
    return bool(rows) and len(set(len(r) for r in rows)) == 1


def text_to_grid(text):
    lines = [line for line in text.strip().split("\n") if line.strip() != ""]
    rows = [_parse_row(line.strip()) for line in lines]
    if any(row is None for row in rows) or not _is_rectangular(rows):
        return None
    return rows


def format_pair_as_text(pair):
    input_grid, output_grid = pair
    return PROMPT_PREFIX + grid_to_text(input_grid) + COMPLETION_PREFIX + grid_to_text(output_grid)


def build_inference_prompt(test_input):
    return PROMPT_PREFIX + grid_to_text(test_input) + COMPLETION_PREFIX


def _augment_pair(pair, transform):
    input_grid, output_grid = pair
    return (transform(input_grid), transform(output_grid))


def build_augmented_pairs(train_pairs, transforms=GEOMETRIC_TRANSFORMS):
    augmented = []
    for pair in train_pairs:
        for transform in transforms:
            augmented.append(_augment_pair(pair, transform))
    return augmented

In [ ]:
import random


def colors_present(pair):
    input_grid, output_grid = pair
    colors = set()
    for grid in (input_grid, output_grid):
        for row in grid:
            colors.update(row)
    return colors


def _random_permutation(colors, rng):
    colors_list = sorted(colors)
    shuffled = colors_list[:]
    rng.shuffle(shuffled)
    return dict(zip(colors_list, shuffled))


def generate_color_variant(pair, rng):
    input_grid, output_grid = pair
    mapping = _random_permutation(colors_present(pair), rng)
    return (apply_color_mapping(input_grid, mapping), apply_color_mapping(output_grid, mapping))


def expand_pairs_with_color_variants(pairs, num_variants, seed):
    rng = random.Random(seed)
    expanded = list(pairs)
    for pair in pairs:
        for _ in range(num_variants):
            expanded.append(generate_color_variant(pair, rng))
    return expanded

In [ ]:
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import torch
from unsloth import FastLanguageModel


def load_base_model(config):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.model_name,
        max_seq_length=config.max_seq_length,
        load_in_4bit=config.load_in_4bit,
    )
    return model, tokenizer


def attach_fresh_lora(model, config):
    return FastLanguageModel.get_peft_model(
        model,
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        lora_dropout=config.lora_dropout,
        target_modules=config.lora_target_modules,
    )


def detach_lora(model):
    model.unload()

In [ ]:
"""ADR 0049 circuit breaker (2026-09-16 port to this notebook's Part C,
mirroring src/evaluation/task_time_limit.py,
src/solvers/neural/deadline_stopping_criteria.py, and
src/solvers/neural/deadline_callback.py exactly). This notebook does not
import from src/, so it must be kept manually in sync (same drift risk
ADR 0032's reactivation note already documented for this Part C port).

NEURAL_TASK_CEILING_SECONDS bounds a single task's own TTT+generation
wall time: if it runs away, that task's neural processing is aborted and
neural_solve_task falls back to the symbolic solver, exactly like any
other neural failure. TIME_LIMIT_ABORTS counts how often this triggers
across a real run.
"""
from dataclasses import dataclass, field
from typing import Callable, List

from transformers import StoppingCriteria, StoppingCriteriaList, TrainerCallback

NEURAL_TASK_CEILING_SECONDS = 400.0


class TaskTimeExceeded(Exception):
    """Raised when a single task's neural processing exceeds its ceiling."""


@dataclass
class TaskTimeLimiter:
    ceiling_seconds: float
    clock: Callable[[], float] = time.monotonic
    _start: float = field(default=0.0, init=False, repr=False)

    def start(self):
        self._start = self.clock()

    def elapsed(self):
        return self.clock() - self._start

    def deadline(self):
        return self._start + self.ceiling_seconds

    def check(self):
        elapsed = self.elapsed()
        if elapsed > self.ceiling_seconds:
            raise TaskTimeExceeded(
                f"neural processing exceeded the {self.ceiling_seconds:.0f}s "
                f"per-task ceiling ({elapsed:.1f}s elapsed)"
            )


@dataclass
class TimeLimitAbortTracker:
    """Records which tasks were aborted by the per-task time ceiling, so a
    real 240-task run can report how often this triggers in practice."""
    aborted_task_ids: List[str] = field(default_factory=list)

    def record_abort(self, task_id):
        self.aborted_task_ids.append(task_id)

    @property
    def count(self):
        return len(self.aborted_task_ids)


TIME_LIMIT_ABORTS = TimeLimitAbortTracker()


class DeadlineStoppingCriteria(StoppingCriteria):
    """Stops a single model.generate() call once a wall-clock deadline
    passes, so one slow completion cannot by itself blow through the
    per-task ceiling regardless of max_new_tokens."""

    def __init__(self, deadline, clock=time.monotonic):
        self._deadline = deadline
        self._clock = clock

    def __call__(self, input_ids, scores, **kwargs):
        return self._clock() >= self._deadline


class DeadlineTrainerCallback(TrainerCallback):
    """HF Trainer callback enforcing the per-task ceiling during TTT, so an
    unusually slow fine-tune cannot by itself exhaust it before generation
    even starts."""

    def __init__(self, limiter):
        self._limiter = limiter

    def on_step_end(self, args, state, control, **kwargs):
        self._limiter.check()
        return control


print(f"ADR 0049 circuit breaker ready: NEURAL_TASK_CEILING_SECONDS={NEURAL_TASK_CEILING_SECONDS}")


In [ ]:
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments


class _TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_seq_length):
        self.encodings = tokenizer(
            texts, truncation=True, max_length=max_seq_length, padding=True, return_tensors="pt"
        )

    def __len__(self):
        return self.encodings["input_ids"].shape[0]

    def __getitem__(self, idx):
        item = {key: tensor[idx] for key, tensor in self.encodings.items()}
        labels = item["input_ids"].clone()
        labels[item["attention_mask"] == 0] = -100
        item["labels"] = labels
        return item


def _build_training_args(config):
    return TrainingArguments(
        output_dir=config.checkpoint_output_dir,
        num_train_epochs=config.ttt_num_epochs,
        learning_rate=config.ttt_learning_rate,
        per_device_train_batch_size=2,
        logging_steps=1,
        save_strategy=config.checkpoint_save_strategy,
        save_steps=config.checkpoint_save_steps,
        save_total_limit=config.checkpoint_save_total_limit,
        report_to=[],
    )


def _texts_with_eos(texts, tokenizer):
    return [text + tokenizer.eos_token for text in texts]


def _build_training_texts(train_pairs, config):
    transforms = GEOMETRIC_TRANSFORMS if config.use_geometric_augmentation else [identity]
    pairs = build_augmented_pairs(train_pairs, transforms)
    if config.use_color_augmentation:
        pairs = expand_pairs_with_color_variants(
            pairs, config.num_color_augmentations_per_pair, config.color_augmentation_seed
        )
    return [format_pair_as_text(pair) for pair in pairs]


def train_on_task(model, tokenizer, train_pairs, config, limiter=None):
    """limiter, when given, aborts training with TaskTimeExceeded once the
    per-task neural time ceiling passes (ADR 0049 circuit breaker)."""
    texts = _texts_with_eos(_build_training_texts(train_pairs, config), tokenizer)
    dataset = _TextDataset(texts, tokenizer, config.max_seq_length)
    callbacks = [DeadlineTrainerCallback(limiter)] if limiter is not None else None
    trainer = Trainer(model=model, args=_build_training_args(config), train_dataset=dataset, callbacks=callbacks)
    torch.cuda.empty_cache()
    trainer.train()
    return model


In [ ]:
import re
from dataclasses import replace

_MAX_SAMPLING_ATTEMPTS_PER_PREDICTION = 3
_SECOND_EXAMPLE_MARKER = "\nInput:"
_MIN_REPEATED_LINE_RUN = 10
_ESCALATED_NO_REPEAT_NGRAM_SIZE = 3
_CODE_MARKERS = ("```", "def ", "import ")
_MIN_NATURAL_LANGUAGE_WORDS = 4
_ALPHA_WORD_PATTERN = re.compile(r"[A-Za-z]+")


def _truncate_before_second_input(text):
    idx = text.find(_SECOND_EXAMPLE_MARKER)
    return text if idx == -1 else text[:idx]


def _has_hallucinated_second_example(raw_completion):
    return _SECOND_EXAMPLE_MARKER in raw_completion


def _has_degenerate_repetition(raw_completion, min_run=_MIN_REPEATED_LINE_RUN):
    lines = raw_completion.splitlines()
    run_length = 1
    for previous, current in zip(lines, lines[1:]):
        if current == previous and current.strip():
            run_length += 1
            if run_length >= min_run:
                return True
        else:
            run_length = 1
    return False


def _has_topic_drift(raw_completion, min_natural_language_words=_MIN_NATURAL_LANGUAGE_WORDS):
    if any(marker in raw_completion for marker in _CODE_MARKERS):
        return True
    for line in raw_completion.splitlines():
        if len(_ALPHA_WORD_PATTERN.findall(line)) >= min_natural_language_words:
            return True
    return False


def _shows_degenerate_pattern(raw_completion):
    return (
        _has_hallucinated_second_example(raw_completion)
        or _has_degenerate_repetition(raw_completion)
        or _has_topic_drift(raw_completion)
    )


def _build_escalated_config(baseline, no_repeat_ngram_size=_ESCALATED_NO_REPEAT_NGRAM_SIZE):
    return replace(baseline, no_repeat_ngram_size=no_repeat_ngram_size)


def _generate_completion(model, tokenizer, prompt, config, seed, limiter=None):
    torch.manual_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    stopping_criteria = None
    if limiter is not None:
        # ADR 0049 circuit breaker: bounds a single generate() call so it
        # cannot by itself blow through the per-task ceiling, regardless of
        # max_new_tokens or how slow this particular completion turns out.
        stopping_criteria = StoppingCriteriaList([DeadlineStoppingCriteria(limiter.deadline())])
    output_ids = model.generate(
        **inputs,
        max_new_tokens=config.max_new_tokens,
        do_sample=True,
        temperature=config.generation_temperature,
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=config.repetition_penalty,
        no_repeat_ngram_size=config.no_repeat_ngram_size,
        stopping_criteria=stopping_criteria,
    )
    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    completion = full_text[len(prompt):]
    if config.stop_on_second_input:
        completion = _truncate_before_second_input(completion)
    return completion


def generate_grid_predictions(model, tokenizer, test_input, config, limiter=None):
    """ADR 0032: every attempt is checked for the degenerate pattern (not
    only the first); on detection, this call's remaining attempts escalate
    one-way to no_repeat_ngram_size=3.

    limiter, when given, raises TaskTimeExceeded between attempts once the
    per-task neural time ceiling passes (ADR 0049 circuit breaker), on top
    of the mid-call stopping criteria _generate_completion already applies.
    """
    prompt = build_inference_prompt(test_input)
    predictions = []
    max_attempts = config.num_predictions * _MAX_SAMPLING_ATTEMPTS_PER_PREDICTION
    active_config = config
    escalated = False
    for seed in range(max_attempts):
        if limiter is not None:
            limiter.check()
        if len(predictions) >= config.num_predictions:
            break
        completion = _generate_completion(model, tokenizer, prompt, active_config, seed, limiter)
        is_degenerate = _shows_degenerate_pattern(completion)
        if not escalated and config.enable_conditional_escalation and is_degenerate:
            escalated = True
            active_config = _build_escalated_config(config)
            print(f"[ADR0032] degenerate pattern detected at attempt {seed}, escalating to no_repeat_ngram_size={active_config.no_repeat_ngram_size}")
        grid = text_to_grid(completion)
        if grid is not None and not is_degenerate and grid not in predictions:
            predictions.append(grid)
    return predictions


In [ ]:
_BASE_MODEL = None
_TOKENIZER = None


def _get_base_model():
    global _BASE_MODEL, _TOKENIZER
    if _BASE_MODEL is None:
        _BASE_MODEL, _TOKENIZER = load_base_model(NEURAL_CONFIG)
        FastLanguageModel.for_inference(_BASE_MODEL)
    return _BASE_MODEL, _TOKENIZER


def _passes_self_consistency(model, tokenizer, train_pairs, config, limiter):
    FastLanguageModel.for_inference(model)
    for input_grid, output_grid in train_pairs:
        limiter.check()
        predictions = generate_grid_predictions(model, tokenizer, input_grid, config, limiter)
        if output_grid not in predictions:
            return False
    return True


def _predict_task_with_neural_solver(task, config, limiter):
    train_pairs = [(p["input"], p["output"]) for p in task["train"]]
    base_model, tokenizer = _get_base_model()
    model = attach_fresh_lora(base_model, config)
    try:
        FastLanguageModel.for_training(model)
        model = train_on_task(model, tokenizer, train_pairs, config, limiter=limiter)
        if not _passes_self_consistency(model, tokenizer, train_pairs, config, limiter):
            return None
        return [generate_grid_predictions(model, tokenizer, p["input"], config, limiter) for p in task["test"]]
    finally:
        detach_lora(model)


def neural_solve_task(task, config=NEURAL_CONFIG, task_id=None):
    """Mirrors src/solvers/neural_solver.py::solve_task, dict-based.

    Falls back to the symbolic solver (Section 4 above) per test pair
    whenever the neural path returns None (self-consistency failed), any
    per-pair prediction list is empty, or the ADR 0049 per-task time
    ceiling is exceeded (TIME_LIMIT_ABORTS records task_id in that case),
    exactly like production.
    """
    train_pairs = [(p["input"], p["output"]) for p in task["train"]]
    limiter = TaskTimeLimiter(ceiling_seconds=NEURAL_TASK_CEILING_SECONDS)
    limiter.start()
    try:
        neural_predictions = _predict_task_with_neural_solver(task, config, limiter)
    except TaskTimeExceeded as exc:
        print(f"Task {task_id} exceeded the neural time ceiling, falling back to symbolic: {exc}")
        TIME_LIMIT_ABORTS.record_abort(task_id)
        neural_predictions = None
    except Exception as exc:
        print(f"Neural solve raised, falling back to symbolic: {type(exc).__name__}: {exc}")
        neural_predictions = None
    if neural_predictions is not None and all(neural_predictions):
        return neural_predictions
    return [predict_test_input(train_pairs, p["input"]) for p in task["test"]]


## Part D - Time-budgeted hybrid pipeline (ADR 0049)

Mirrors `src/evaluation/time_budget.py`, `task_ordering.py`, and
`build_hybrid_submission.py`: the symbolic solver runs on every task
first as the guaranteed ADR 0011 fallback, then the neural solver is
attempted on tasks in ascending expected-cost order (total grid cell
count, a cheap GPU-free proxy) only while the time budget is not yet
exhausted; any exception, timeout, or empty result keeps the symbolic
answer untouched.

`NEURAL_PASS_CEILING_SECONDS` below is ADR 0049's original 8h starting
point, explicitly a placeholder to be recalculated in Part E once real
L4x4 timing is measured on this hardware - it must not be trusted until
then.

In [ ]:
TOTAL_KAGGLE_BUDGET_SECONDS = 12 * 3600
SAFETY_MARGIN_SECONDS = 4 * 3600
NEURAL_PASS_CEILING_SECONDS = TOTAL_KAGGLE_BUDGET_SECONDS - SAFETY_MARGIN_SECONDS  # ADR 0049 placeholder, 8h


@dataclass
class TimeBudget:
    ceiling_seconds: float
    clock: object = time.monotonic
    _elapsed_seconds: float = field(default=0.0, init=False, repr=False)

    def record(self, seconds):
        self._elapsed_seconds += max(seconds, 0.0)

    def is_exhausted(self):
        return self._elapsed_seconds >= self.ceiling_seconds

    def remaining_seconds(self):
        return max(self.ceiling_seconds - self._elapsed_seconds, 0.0)

    def elapsed_seconds(self):
        return self._elapsed_seconds


def _grid_cells(grid):
    return len(grid) * len(grid[0]) if grid else 0


def _task_total_cells(task):
    train_cells = sum(_grid_cells(p["input"]) + _grid_cells(p["output"]) for p in task["train"])
    test_cells = sum(_grid_cells(p["input"]) for p in task["test"])
    return train_cells + test_cells


def order_tasks_by_expected_neural_cost(tasks):
    return sorted(tasks, key=lambda task_id: _task_total_cells(tasks[task_id]))

In [ ]:
def _attempt_neural_task(task, task_id, config, budget):
    start = budget.clock()
    try:
        predictions = neural_solve_task(task, config, task_id=task_id)
    except Exception as exc:
        print(f"  neural attempt raised: {type(exc).__name__}: {exc}")
        predictions = None
    finally:
        budget.record(budget.clock() - start)
    if not predictions or not all(predictions):
        return None
    return predictions


def run_hybrid_pass(tasks, config, budget):
    """Symbolic pass over all tasks first, then a budget-gated neural overlay.

    Returns {task_id: predictions_per_test_pair}. Never raises for an
    individual task; a failed/timed-out/empty neural attempt (including an
    ADR 0049 time-ceiling abort) keeps the symbolic answer.
    """
    predictions_by_task = {task_id: solve_task(task) for task_id, task in tasks.items()}

    ordered_ids = order_tasks_by_expected_neural_cost(tasks)
    for task_id in ordered_ids:
        if budget.is_exhausted():
            print(f"Time budget exhausted before task {task_id}; keeping symbolic answers for the rest.")
            break
        neural_predictions = _attempt_neural_task(tasks[task_id], task_id, config, budget)
        if neural_predictions is not None:
            predictions_by_task[task_id] = neural_predictions

    return predictions_by_task


def build_submission_from_predictions(predictions_by_task, tasks):
    submission = {}
    for task_id, task in tasks.items():
        train_pairs = [(p["input"], p["output"]) for p in task["train"]]
        predictions_per_pair = predictions_by_task[task_id]
        submission[task_id] = [
            _build_attempt_pair(predictions, test_pair["input"], train_pairs)
            for predictions, test_pair in zip(predictions_per_pair, task["test"])
        ]
    return submission


## Part E - Real-GPU timing calibration (ADR 0049 intermediate test)

**Goal:** measure real per-task wall time for the *full* neural path
(fresh LoRA attach + TTT + self-consistency + generation) on this
kernel's actual GPU hardware, across a small but *varied* sample,
before ever running the full 240-task hybrid pipeline against the
scarce, shared weekly GPU quota.

**History:** the first real run of this cell (2026-09-16) measured only
2 tasks (`0d3d703e`, `25d8a9c8`, both 81 cells, both on the
ascending-expected-cost front of the queue) across 2 separate pushes,
94.66s/task and 86.31s/task. That is too small and too narrow a sample
to trust as representative on its own (ADR 0028 documented up to a 16x
slowdown on individual tasks from a generation-retry-loop failure mode,
`13e47133`/`0934a4d8`, driven by task content, not grid size). A second
round added 6 more tasks chosen for grid-size and structural diversity
and found the picture materially worse: combined mean 187.57s/task,
worst observed 886.35s/task on `264363fd` (6300 cells), about 9.8x that
round's own mean and 3.3x slower than the near-identical-size
`40f6cd08` (6120 cells, 266.71s). That outlier's retry ratio was
*negative*, meaning the retry-count anomaly detector below did NOT flag
it, an honest detection-gap finding: the slowdown is not explained by
ADR 0028's known retry-loop pattern.

**2026-09-16 outlier isolation test (this round):** direct code reading
found ADR 0032's per-attempt conditional decode-escalation mitigation
was never actually wired into this notebook's production
`generate_grid_predictions` (same root cause as `src/solvers/neural/
generation.py`, see the Part C markdown above for the full root-cause
narrative, including a real circular-import obstacle that made a naive
fix impossible without first extracting the shared marker constant).
That gap is now closed in Part C. This round re-tests `264363fd` with
the mitigation active, listed **first** in `CALIBRATION_TASK_IDS` below
so its isolated result is printed and can be judged before looking at
the full 6-task recalibration that follows in the same run (avoids a
second GPU-quota-consuming push just to re-order it):
- If `264363fd`'s time drops back to near the round's mean (roughly
  90-190s), that confirms the outlier was fully explained by the
  mitigation being inactive, a regression now fixed.
- If it stays high, that confirms a genuinely different, still
  undiagnosed mechanism (most likely content-dependent TTT/generation
  latency unrelated to decode repetition/hallucination), which would
  need its own ADR-0028-style investigation before any budget decision,
  and the 240-task projection below should NOT be treated as resolved
  in that case.

**Note:** `13e47133` and `0934a4d8` (ADR 0028's original anomalous
tasks) are confirmed, by direct lookup, **not present** in this
competition's real 240-task test set, so they cannot be included
directly. As the closest available proxy, this round includes the two
real test tasks with the lowest train-pair output/input cell ratio
(`239be575` at 0.028, `137eaa0f` at 0.074), the same structural trait
ADR 0028 measured for `0934a4d8` (ratio 0.04) - an extraction-shaped
task whose output is much smaller than its input, historically
associated with hallucinated-continuation generation failures.

Tasks are **not** reselected via the ascending-cost queue this round,
since that queue alone produced the too-narrow original sample; they
are hand-picked across the full cost range actually present in the real
test set (81 to 6300 cells) plus the two extraction-like tasks above.

This cell also prints whether ADR 0032's per-attempt conditional
`no_repeat_ngram_size` decode-escalation mitigation is enabled in this
notebook's `NEURAL_CONFIG` (`enable_conditional_escalation`, now
reactivated, default `True`); actual firing on a given attempt is only
confirmed by the `[ADR0032] degenerate pattern detected...` print
inside `generate_grid_predictions` itself (Part C), which is the real
runtime evidence this round is looking for, not just the static config
value. It also instruments the low-level generation call count per task
so a retry-loop anomaly (ADR 0028's failure signature) can still be
detected directly from real, unlabeled competition test tasks, without
needing ground truth, for tasks the mitigation does not fully resolve.

**This cell does not write or overwrite `submission.json`** (that file,
from Section 6 above, is already the validated symbolic-only
submission); it only prints timing and diagnostics.


In [ ]:
# Historical Part E measurements already made for real on Kaggle GPU
# hardware (2026-09-16), see ADR 0049's "Real Kaggle timing run" and
# "Intermediate calibration test, second round" sections.
# Not re-run here: repeating them with no code/config change would just
# reproduce the known result and burn scarce, shared GPU quota for no new
# information.
HISTORICAL_TIMING_SECONDS = {
    "0d3d703e": [145.01, 130.29],
    "25d8a9c8": [44.30, 42.33],
}

# Per-task times from round 3 (2026-09-16, ADR 0032 active, no per-task time
# ceiling yet), kept for direct before/after comparison against this round's
# re-test, not folded into HISTORICAL_TIMING_SECONDS since every one of
# these 6 tasks is re-run again below.
PRIOR_ROUND_SECONDS = {
    "264363fd": 490.29,
    "3c9b0459": 54.91,
    "239be575": 63.86,
    "137eaa0f": 40.15,
    "2bcee788": 79.37,
    "40f6cd08": 248.18,
}

# 264363fd listed first: round 3 left it as a confirmed, still-undiagnosed
# outlier at 490.29s, above NEURAL_TASK_CEILING_SECONDS=400.0. This round's
# purpose (ADR 0049 circuit breaker test) is to confirm it gets aborted at
# roughly the ceiling and falls back to the symbolic solver instead of
# running to 490s+ again, and that none of the other 5, normal-speed tasks
# are falsely aborted.
CALIBRATION_TASK_IDS = [
    "264363fd",   # 6300 cells, ratio 1.0 - expected to trip the circuit breaker this round
    "3c9b0459",   # 81 cells, ratio 1.0 - small control, same size class as the 2 historical tasks
    "239be575",   # 324 cells, ratio 0.028 - closest available proxy for 0934a4d8's hallucination-risk profile
    "137eaa0f",   # 511 cells, ratio 0.074 - second hallucination-risk proxy
    "2bcee788",   # 900 cells, ratio 1.0 - median-size control
    "40f6cd08",   # 6120 cells, ratio 1.0 - large stress test
]

LOCAL_RTX_4060TI_SECONDS_PER_TASK = 269.92  # ADR 0034, for comparison only, different hardware
NUM_TASKS_TOTAL = len(tasks)

print(
    f"ADR 0049 per-task time circuit breaker: "
    f"NEURAL_TASK_CEILING_SECONDS={NEURAL_TASK_CEILING_SECONDS}s "
    f"(real firing, if any, is confirmed below by the per-task abort message "
    f"and TIME_LIMIT_ABORTS)"
)

# Lightweight retry-rate instrumentation (Part E only, restored afterward,
# does not touch the production Part C code): counts every low-level
# generation call so the ADR 0028 failure signature (the self-consistency/
# generation retry loop repeatedly re-sampling past the theoretical per-task
# minimum) can be detected directly on these real, unlabeled test tasks,
# with no ground truth needed.
_generation_call_count = {"n": 0}
_original_generate_completion = _generate_completion


def _counting_generate_completion(model, tokenizer, prompt, config, seed, limiter=None):
    _generation_call_count["n"] += 1
    return _original_generate_completion(model, tokenizer, prompt, config, seed, limiter)


_generate_completion = _counting_generate_completion

_results = []
for _task_id in CALIBRATION_TASK_IDS:
    _task = tasks[_task_id]
    _min_expected_calls = NEURAL_CONFIG.num_predictions * (len(_task["train"]) + len(_task["test"]))
    _generation_call_count["n"] = 0
    _aborts_before = TIME_LIMIT_ABORTS.count
    print(f"\n--- Timing task {_task_id} (cells={_task_total_cells(_task)}) ---")
    _start = time.monotonic()
    _predictions = neural_solve_task(_task, NEURAL_CONFIG, task_id=_task_id)
    _elapsed = time.monotonic() - _start
    _actual_calls = _generation_call_count["n"]
    _was_aborted = TIME_LIMIT_ABORTS.count > _aborts_before
    _retry_ratio = (_actual_calls / _min_expected_calls) - 1.0 if _min_expected_calls else 0.0
    _anomaly = _retry_ratio > 0.5
    _results.append((_task_id, _elapsed, _actual_calls, _min_expected_calls, _retry_ratio, _anomaly, _was_aborted))
    _prior = PRIOR_ROUND_SECONDS.get(_task_id)
    _delta_note = f", round 3 (no ceiling) was {_prior:.2f}s" if _prior is not None else ""
    print(
        f"Task {_task_id}: {_elapsed:.2f}s, generation calls={_actual_calls} "
        f"(min expected {_min_expected_calls}, retry overhead {_retry_ratio:+.0%})"
        f"{_delta_note}"
        + ("  <-- ANOMALY (excess retries, degenerate/hallucination pattern)" if _anomaly else "")
        + ("  <-- ABORTED BY CIRCUIT BREAKER" if _was_aborted else "")
    )
    if _task_id == "264363fd":
        print(
            "\n=== Circuit breaker verdict (264363fd, ADR 0049 2026-09-16 round 4 test) ===\n"
            f"New time: {_elapsed:.2f}s vs. round 3 (no ceiling) 490.29s. "
            f"Aborted by the per-task ceiling: {_was_aborted}.\n"
            "This does NOT resolve 264363fd's underlying still-undiagnosed slowdown "
            "mechanism (see round 3's verdict); it only bounds how much of the shared "
            "neural-pass budget this or any similarly-affected task can consume."
        )

_generate_completion = _original_generate_completion  # restore the production function

_new_seconds = [r[1] for r in _results]
_all_measurements = list(_new_seconds)
for _hist_list in HISTORICAL_TIMING_SECONDS.values():
    _all_measurements.extend(_hist_list)

_mean_seconds = sum(_all_measurements) / len(_all_measurements)
_worst_seconds = max(_all_measurements)
_num_distinct_tasks = len(_new_seconds) + len(HISTORICAL_TIMING_SECONDS)

print("\n=== Calibration summary (round 4, ADR 0049 circuit breaker active) ===")
print(
    f"New tasks this round: {len(_new_seconds)}, historical tasks: {len(HISTORICAL_TIMING_SECONDS)} "
    f"({sum(len(v) for v in HISTORICAL_TIMING_SECONDS.values())} historical measurements)"
)
print(f"Combined sample: {len(_all_measurements)} measurements over {_num_distinct_tasks} distinct tasks")
print(f"Mean: {_mean_seconds:.2f}s/task")
print(f"Worst observed: {_worst_seconds:.2f}s/task")
print(f"Anomalies flagged this round: {[r[0] for r in _results if r[5]]}")
print(f"Aborted by the circuit breaker this round: {[r[0] for r in _results if r[6]]}")
print(f"TIME_LIMIT_ABORTS total count: {TIME_LIMIT_ABORTS.count}, ids: {TIME_LIMIT_ABORTS.aborted_task_ids}")

print(f"\nLocal RTX 4060 Ti reference (ADR 0034): {LOCAL_RTX_4060TI_SECONDS_PER_TASK:.2f}s/task")

_mean_projection_seconds = _mean_seconds * NUM_TASKS_TOTAL
_worst_projection_seconds = _worst_seconds * NUM_TASKS_TOTAL
_ceiling_bound_projection_seconds = NEURAL_TASK_CEILING_SECONDS * NUM_TASKS_TOTAL
print(f"\nFull-run projections ({NUM_TASKS_TOTAL} tasks, no parallelism assumed):")
print(f"  mean-based:        {_mean_projection_seconds:.0f}s (~{_mean_projection_seconds / 3600:.2f}h)")
print(f"  worst-based:       {_worst_projection_seconds:.0f}s (~{_worst_projection_seconds / 3600:.2f}h)")
print(
    f"  ceiling-bound worst-case (every task hits NEURAL_TASK_CEILING_SECONDS): "
    f"{_ceiling_bound_projection_seconds:.0f}s (~{_ceiling_bound_projection_seconds / 3600:.2f}h)"
)

print("\nThis result must be reported to the user and the ADR 0049 budget recalculated")
print("BEFORE proposing or enabling the full hybrid pipeline in Part F below.")


## Part F - Full hybrid pipeline (2026-09-17: writes directly to submission.json)

**Design change (2026-09-17):** a real submission attempt of kernel v8's
`submission_hybrid.json` was rejected by Kaggle with `HTTP 400`: the
Code Competition submission API requires the submitted file to be named
exactly `submission.json`, regardless of the kernel's actual output
filename. This cell now **overwrites `submission.json`** (the same
`OUTPUT_PATH` Section 6 already wrote, symbolic-only) with the hybrid
pipeline's result, so the file that gets submitted is the one Kaggle
actually accepts. A secondary copy is also written to
`submission_hybrid.json` as an internal record/comparison, unchanged
from the prior design; it is not what gets submitted. See ADR 0049's
"Design change: Part F writes to submission.json" section for full
detail.

**`RUN_FULL_HYBRID_PIPELINE` was flipped to `True` below after:**
1. Part E's real timing was reported back to the user across 4 rounds
   (see ADR 0049's "Genuine round 4, real circuit-breaker evidence"
   section: mean 127.25s/task, worst 400.07s/task, `264363fd` confirmed
   correctly aborted by the circuit breaker with zero false positives),
2. a first full 240-task dry run (kernel v8, 2026-09-17) completed
   clean and was validated, but the submission attempt itself was
   rejected for the filename reason above, and
3. the user reviewed that result and gave separate, explicit approval
   in chat for this corrected run, still a **dry run only**: generate
   and review the real `submission.json`, do **not** submit to the
   leaderboard yet.

This run consumes up to ~8h of the shared weekly T4 GPU quota
(30h/week). Flipping this flag runs the hybrid pipeline and
**overwrites the existing `submission.json`** inside this kernel run
(the already-submitted symbolic-only version, ref 56256382, is
unaffected outside this kernel and stays the project's real leaderboard
submission until a fresh, separate submission approval is given).
**It still never calls the Kaggle submission API** - that is a
permanent, separate constraint that no flag in this notebook controls;
no real leaderboard submission happens at any stage without the user's
own separate, explicit approval given in chat, independent of this
dry-run approval.


In [ ]:
RUN_FULL_HYBRID_PIPELINE = True  # 2026-09-17: explicit user approval given in chat for a dry run; Part F now overwrites submission.json directly (Kaggle requires this exact filename for competition_submit_code), NOT submitted to the leaderboard by this notebook; see docs/decisions/0049-pipeline-hibrido-orcamento-tempo.md "Design change: Part F writes to submission.json" section

if RUN_FULL_HYBRID_PIPELINE:
    hybrid_budget = TimeBudget(ceiling_seconds=NEURAL_PASS_CEILING_SECONDS)

    _hybrid_start = time.monotonic()
    hybrid_predictions = run_hybrid_pass(tasks, NEURAL_CONFIG, hybrid_budget)
    _hybrid_elapsed = time.monotonic() - _hybrid_start

    hybrid_submission = build_submission_from_predictions(hybrid_predictions, tasks)
    validate_submission(hybrid_submission, tasks)

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(hybrid_submission, f)

    HYBRID_OUTPUT_PATH = (
        "submission_hybrid.json" if not os.path.isdir("/kaggle/working")
        else "/kaggle/working/submission_hybrid.json"
    )
    with open(HYBRID_OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(hybrid_submission, f)

    print(f"Hybrid pipeline done in {_hybrid_elapsed:.1f}s, neural budget used {hybrid_budget.elapsed_seconds():.1f}s")
    print(f"Written to {OUTPUT_PATH} (overwrites the symbolic-only submission from Section 6; this is the file Kaggle requires for submission)")
    print(f"Also written to {HYBRID_OUTPUT_PATH} as a secondary internal record/comparison copy")
    print("Reminder: this notebook does NOT call the Kaggle submission API.")
else:
    print("RUN_FULL_HYBRID_PIPELINE is False, skipping the full hybrid run.")
    print("Enable it only after Part E's timing is reported and reviewed, and with explicit user approval.")
